# Kandinsky Game→Audio — Step 4: Thread Game Tokens Through the Full DiT

This notebook extends Step 3 from one real fused decoder block to Kandinsky's real `DiffusionTransformer3D` forward path.

It adds optional game tokens to the model API and forwards them through both:
- audio-only generation, where the video stream is absent;
- joint video+audio generation, where Kandinsky uses fused blocks.

The model is deliberately tiny and randomly initialized. **No pretrained weights, training, audio generation, or pipeline configuration are used.**

## What we are checking

1. The patched model accepts typed game tokens and their validity mask.
2. Existing model weights load into the expanded model; only new Game→Audio parameters are missing.
3. At the zero-initialized gate, the expanded model matches the unmodified model.
4. Opening the final block's gate changes audio while preserving the direct video output in that same last block.
5. Masked game tokens do not affect audio.

A later block can carry game-influenced audio back into video through Kandinsky's existing Audio→Video attention. This notebook opens only the **last** block to isolate the direct Game→Audio path.

## Data flow

`game_tokens [batch, events, game_dim]` → Game→Audio cross-attention → gated audio hidden states → Kandinsky audio output

The event encoder is outside this step. The notebook uses synthetic vectors to verify the interface and propagation only.

In [ ]:
import importlib.util
import subprocess
import sys
import types
from pathlib import Path

import torch
import torch.nn.functional as F

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device: CPU is sufficient for this small forward-path check.")

## 1. Fetch the exact Kandinsky source revision

We use the same public source commit as Steps 1–3. The patch is written to a separate file inside the Colab runtime; the cloned source file remains unchanged.

In [ ]:
K6_COMMIT = "7ceeefad7fb625011f7db602c43893a989f19c15"
K6_DIR = Path("/content/kandinsky-6-step4-lab")

if not K6_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/kandinskylab/kandinsky-6.git", str(K6_DIR)
    ], check=True)

current = subprocess.check_output(
    ["git", "-C", str(K6_DIR), "rev-parse", "HEAD"], text=True
).strip()
if current != K6_COMMIT:
    subprocess.run(
        ["git", "-C", str(K6_DIR), "fetch", "--depth", "1", "origin", K6_COMMIT],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(K6_DIR), "checkout", "--detach", K6_COMMIT],
        check=True,
    )

DIT_PATH = K6_DIR / "kandinsky/core/components/dit.py"
source = DIT_PATH.read_text(encoding="utf-8")
assert subprocess.check_output(
    ["git", "-C", str(K6_DIR), "rev-parse", "HEAD"], text=True
).strip() == K6_COMMIT
print("Pinned source:", K6_COMMIT)
print("Original source:", DIT_PATH)

## 2. Patch a scratch copy

Every edit below targets a unique source anchor. If Kandinsky's source no longer matches the pinned revision, the notebook stops instead of applying a guessed patch.

The new branch casts incoming game tokens to the attention layer's device and weight dtype. This matters because an event encoder may emit float32 while a deployed DiT uses lower-precision weights.

In [ ]:
def replace_once(text, old, new, label):
    count = text.count(old)
    if count != 1:
        raise RuntimeError(f"Expected one source anchor for {label}; found {count}.")
    return text.replace(old, new, 1)

patched = source

# Add the optional event-token width to the real fused block.
patched = replace_once(
    patched,
    "        ca_rope: bool = False, cross_gates: bool = False, fix_modulation: bool = False,\n    ):\n",
    "        ca_rope: bool = False, cross_gates: bool = False, fix_modulation: bool = False,\n"
    "        game_dim: int | None = None,\n    ):\n",
    "fused-block constructor",
)

# Create the Game→Audio attention branch only when game_dim is configured.
patched = replace_once(
    patched,
    "        self.model_dim     = model_dim\n        self.model_dim_a   = model_dim_a\n",
    "        self.model_dim     = model_dim\n        self.model_dim_a   = model_dim_a\n"
    "        self.game_dim      = game_dim\n"
    "        if game_dim is None:\n"
    "            self.game_audio_norm = None\n"
    "            self.game_to_audio = None\n"
    "            self.game_audio_modulation = None\n"
    "        else:\n"
    "            self.game_audio_norm = nn.LayerNorm(model_dim_a, elementwise_affine=False)\n"
    "            self.game_to_audio = MultiheadCrossAttention(model_dim_a, head_dim_a, game_dim, engine)\n"
    "            self.game_audio_modulation = Modulation(time_dim_a, model_dim_a, 1)\n",
    "fused-block Game→Audio modules",
)

# Extend the fused block call. These are also used for audio-only calls.
patched = replace_once(
    patched,
    "        av_gate_scale: float = 1.0,\n"
    "        va_gate_scale: float = 1.0,\n"
    "    ):\n",
    "        av_gate_scale: float = 1.0,\n"
    "        va_gate_scale: float = 1.0,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ):\n",
    "fused-block forward signature",
)

# Apply the branch after existing text/AV updates and before the audio FFN.
patched = replace_once(
    patched,
    "        else:\n"
    "            vis = apply_gate_sum(vis, vis_out_t, gate_v).type_as(vis)\n"
    "\n"
    "        # ---- FFN ----\n",
    "        else:\n"
    "            vis = apply_gate_sum(vis, vis_out_t, gate_v).type_as(vis)\n"
    "\n"
    "        # Game events condition the audio stream in both audio-only and AV calls.\n"
    "        if game_tokens is not None:\n"
    "            if aud is None:\n"
    "                raise ValueError(\"Game conditioning requires an audio stream.\")\n"
    "            if self.game_to_audio is None:\n"
    "                raise ValueError(\"Construct this block with game_dim to accept game_tokens.\")\n"
    "            if game_tokens.ndim != 3 or game_tokens.shape[-1] != self.game_dim:\n"
    "                raise ValueError(\"game_tokens must have shape [batch, tokens, game_dim].\")\n"
    "            if game_tokens.shape[0] != aud.shape[0]:\n"
    "                raise ValueError(\"game_tokens and audio must have the same batch size.\")\n"
    "            game_tokens = game_tokens.to(\n"
    "                device=aud.device, dtype=self.game_to_audio.to_key.weight.dtype\n"
    "            )\n"
    "            if game_attn_mask is not None:\n"
    "                game_attn_mask = game_attn_mask.to(device=aud.device, dtype=torch.bool)\n"
    "            game_query = self.game_audio_norm(aud).type_as(aud)\n"
    "            game_update = self.game_to_audio(game_query, game_tokens, game_attn_mask)\n"
    "            game_gate = self.game_audio_modulation(t_a)\n"
    "            aud = apply_gate_sum(aud, game_update, game_gate).type_as(aud)\n"
    "\n"
    "        # ---- FFN ----\n",
    "fused-block Game→Audio insertion",
)

# Add game_dim to DiffusionTransformer3D and pass it to every fused block.
patched = replace_once(
    patched,
    "        visual_token_type_num_embeddings: int = 0,\n"
    "    ):\n",
    "        visual_token_type_num_embeddings: int = 0,\n"
    "        game_dim: int | None = None,\n"
    "    ):\n",
    "DiT constructor signature",
)
patched = replace_once(
    patched,
    "        self.is_multimodal  = is_multimodal\n"
    "        self.in_visual_dim  = in_visual_dim\n",
    "        self.is_multimodal  = is_multimodal\n"
    "        self.game_dim       = game_dim\n"
    "        if game_dim is not None and not is_multimodal:\n"
    "            raise ValueError(\"game_dim is supported only by the multimodal audio model.\")\n"
    "        self.in_visual_dim  = in_visual_dim\n",
    "DiT game_dim configuration",
)
patched = replace_once(
    patched,
    "                    ca_rope=ca_rope, cross_gates=cross_gates, fix_modulation=fix_modulation,\n",
    "                    ca_rope=ca_rope, cross_gates=cross_gates, fix_modulation=fix_modulation,\n"
    "                    game_dim=game_dim,\n",
    "pass game_dim to fused-block stack",
)

# Thread game tokens through the audio-only fused-block loop.
patched = replace_once(
    patched,
    "        aud_rope: Tensor | None,\n"
    "        attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor | None, Tensor | None]:\n",
    "        aud_rope: Tensor | None,\n"
    "        attn_mask: Tensor | None = None,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor | None, Tensor | None]:\n",
    "single-modality block-loop signature",
)
patched = replace_once(
    patched,
    "                    _, aud_embed = blk(\n"
    "                        None, aud_embed, te, te, (tm, tm), None, aud_rope, attn_mask,\n"
    "                    )\n",
    "                    _, aud_embed = blk(\n"
    "                        None, aud_embed, te, te, (tm, tm), None, aud_rope, attn_mask,\n"
    "                        game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "                    )\n",
    "audio-only fused-block call",
)

# Thread the same tensors through the joint video+audio loop.
patched = replace_once(
    patched,
    "        aud_rope: Tensor,\n"
    "        attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor, Tensor]:\n",
    "        aud_rope: Tensor,\n"
    "        attn_mask: Tensor | None = None,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor, Tensor]:\n",
    "fused block-loop signature",
)
patched = replace_once(
    patched,
    "                vis_rope, aud_rope,\n"
    "                attn_mask,\n"
    "            )\n"
    "        return vis_embed, aud_embed\n",
    "                vis_rope, aud_rope,\n"
    "                attn_mask,\n"
    "                game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "            )\n"
    "        return vis_embed, aud_embed\n",
    "joint fused-block call",
)

# Add public forward arguments and validate their basic contract.
patched = replace_once(
    patched,
    "        attention_mask: Tensor | None = None,\n"
    "        visual_token_type_ids: Tensor | None = None,\n"
    "    ) -> Tensor | tuple[Tensor, Tensor]:\n",
    "        attention_mask: Tensor | None = None,\n"
    "        visual_token_type_ids: Tensor | None = None,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ) -> Tensor | tuple[Tensor, Tensor]:\n",
    "public forward signature",
)
patched = replace_once(
    patched,
    "        both = x_video is not None and x_audio is not None and self.is_multimodal\n"
    "        attn_mask = self._normalize_attn_mask(attention_mask)\n",
    "        both = x_video is not None and x_audio is not None and self.is_multimodal\n"
    "        if game_attn_mask is not None and game_tokens is None:\n"
    "            raise ValueError(\"game_attn_mask requires game_tokens.\")\n"
    "        if game_tokens is not None:\n"
    "            if not self.is_multimodal or self.game_dim is None:\n"
    "                raise ValueError(\"Configure game_dim on a multimodal model before passing game_tokens.\")\n"
    "            if x_audio is None:\n"
    "                raise ValueError(\"Game conditioning requires x_audio.\")\n"
    "            if game_tokens.ndim != 3 or game_tokens.shape[-1] != self.game_dim:\n"
    "                raise ValueError(\"game_tokens must have shape [batch, events, game_dim].\")\n"
    "            audio_batch = x_audio.shape[0] if x_audio.ndim == 3 else 1\n"
    "            if game_tokens.shape[0] != audio_batch:\n"
    "                raise ValueError(\"game_tokens and x_audio must have the same batch size.\")\n"
    "            if game_attn_mask is not None:\n"
    "                if game_attn_mask.dtype != torch.bool:\n"
    "                    raise ValueError(\"game_attn_mask must be boolean; True means a valid event token.\")\n"
    "                if game_attn_mask.shape[-1] != game_tokens.shape[1]:\n"
    "                    raise ValueError(\"The final game_attn_mask dimension must match the event count.\")\n"
    "        attn_mask = self._normalize_attn_mask(attention_mask)\n",
    "forward argument validation",
)

# Pass the optional condition to audio-only and joint forward paths.
patched = replace_once(
    patched,
    "                None, aud_embed, te, tm, None, aud_rope, attn_mask,\n"
    "            )\n"
    "            return self._project_audio(aud_embed, tm)\n",
    "                None, aud_embed, te, tm, None, aud_rope, attn_mask,\n"
    "                game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "            )\n"
    "            return self._project_audio(aud_embed, tm)\n",
    "audio-only DiT call",
)
patched = replace_once(
    patched,
    "            vis_rope, aud_rope, attn_mask,\n"
    "        )\n"
    "        return self._project_fused(\n",
    "            vis_rope, aud_rope, attn_mask,\n"
    "            game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "        )\n"
    "        return self._project_fused(\n",
    "joint DiT call",
)

PATCH_PATH = DIT_PATH.with_name("dit_game_full.py")
PATCH_PATH.write_text(patched, encoding="utf-8")
print("Patched scratch copy:", PATCH_PATH)
print("Original source unchanged:", DIT_PATH.read_text(encoding="utf-8") == source)

## 3. Import the pinned source without installing the full generation stack

The DiT's actual component code is imported from the pinned repository. We provide only the lightweight SDPA engine adapter used by Kandinsky's attention dispatch. No optional SageAttention or FlashAttention kernel is needed for this CPU-sized demonstration.

In [ ]:
if "kandinsky" in sys.modules:
    raise RuntimeError(
        "A Kandinsky package is already imported. Restart the runtime, then run all cells from the top."
    )

def register_package(name, path):
    module = types.ModuleType(name)
    module.__path__ = [str(path)]
    module.__package__ = name
    sys.modules[name] = module

register_package("kandinsky", K6_DIR / "kandinsky")
register_package("kandinsky.core", K6_DIR / "kandinsky/core")
register_package("kandinsky.core.components", K6_DIR / "kandinsky/core/components")
register_package("kandinsky.core.components.attention", K6_DIR / "kandinsky/core/components/attention")
register_package("kandinsky.runtime", K6_DIR / "kandinsky/runtime")
register_package("kandinsky.runtime.kernels", K6_DIR / "kandinsky/runtime/kernels")

def pinned_sdpa(q, k, v, attn_mask=None):
    # Kandinsky attention tensors are [batch, sequence, heads, head_dim].
    q = q.transpose(1, 2).contiguous()
    k = k.transpose(1, 2).contiguous()
    v = v.transpose(1, 2).contiguous()
    return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask).transpose(1, 2).contiguous()

class SDPAOnlyEngine:
    def __init__(self, engine="auto"):
        if engine not in {"auto", "sdpa"}:
            raise ValueError("This notebook supports only the SDPA attention path.")
    def get_attention(self):
        return pinned_sdpa

engine_module = types.ModuleType("kandinsky.runtime.kernels.attention_engine")
engine_module._sdpa = pinned_sdpa
engine_module.SelfAttentionEngine = SDPAOnlyEngine
engine_module.resolve_attention_engine = lambda requested: "sdpa"
sys.modules[engine_module.__name__] = engine_module

def load_dit_module(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

base_dit = load_dit_module(
    "kandinsky.core.components.dit_base", DIT_PATH
)
game_dit = load_dit_module(
    "kandinsky.core.components.dit_game_full", PATCH_PATH
)
print("Baseline model class:", base_dit.DiffusionTransformer3D)
print("Patched model class:", game_dit.DiffusionTransformer3D)

## 4. Build small baseline and expanded models

These dimensions are only for a fast forward-path check. Both models have identical common layers. The expanded model adds one gated branch per fused block.

In [ ]:
torch.manual_seed(123)

BASE_CONFIG = dict(
    in_visual_dim=4,
    out_visual_dim=4,
    in_text_dim=12,
    in_text_dim2=6,
    time_dim=16,
    patch_size=(1, 2, 2),
    model_dim=24,
    ff_dim=48,
    num_text_blocks=1,
    num_visual_blocks=2,
    axes_dims=(2, 2, 2),
    visual_cond=False,
    is_multimodal=True,
    in_audio_dim=5,
    out_audio_dim=5,
    model_dim_a=24,
    time_dim_a=16,
    ff_dim_a=48,
    axes_dims_a=(2, 2, 2),
    attention_engine="sdpa",
)
GAME_DIM = 7

baseline = base_dit.DiffusionTransformer3D(**BASE_CONFIG).eval()
expanded = game_dit.DiffusionTransformer3D(
    **BASE_CONFIG, game_dim=GAME_DIM
).eval()

load_result = expanded.load_state_dict(baseline.state_dict(), strict=False)
assert not load_result.unexpected_keys
assert load_result.missing_keys
assert all(
    ("game_audio_norm" in key)
    or ("game_to_audio" in key)
    or ("game_audio_modulation" in key)
    for key in load_result.missing_keys
), load_result.missing_keys

print("Common baseline weights copied into expanded model.")
print("New Game→Audio parameters:", len(load_result.missing_keys))
for key in load_result.missing_keys[:8]:
    print(" -", key)
print("All missing keys belong to the new branch: passed")

## 5. Create miniature inputs

The game-token vectors are float64 on purpose. The patched attention branch should safely cast them to match its own device and weight dtype. The event-validity mask is boolean; `True` marks a real token and `False` marks padding.

In [ ]:
BATCH = 2
AUDIO_LEN = 8
TEXT_LEN = 4
GAME_LEN = 5

x_audio = torch.randn(BATCH, AUDIO_LEN, BASE_CONFIG["in_audio_dim"])
audio_text = torch.randn(BATCH, TEXT_LEN, BASE_CONFIG["in_text_dim"])
audio_pooled = torch.randn(BATCH, BASE_CONFIG["in_text_dim2"])
audio_time = torch.tensor([0.15, 0.8])

# float64 verifies the model-side dtype conversion on incoming game features.
game_tokens = torch.randn(BATCH, GAME_LEN, GAME_DIM, dtype=torch.float64)
game_valid = torch.tensor([
    [True, True, True, False, False],
    [True, True, False, False, False],
])
game_attn_mask = game_valid[:, None, None, :]

audio_positions = torch.arange(AUDIO_LEN)
text_positions = torch.arange(TEXT_LEN)
audio_rope = expanded.audio_rope(audio_positions)
audio_text_rope = expanded.audio_text_rope(text_positions)

print("Audio latent:", tuple(x_audio.shape))
print("Game token input:", tuple(game_tokens.shape), game_tokens.dtype)
print("Game mask:", tuple(game_attn_mask.shape), game_attn_mask.dtype)

## 6. Audio-only route through the full model

This follows Kandinsky's multimodal audio-only path: text encoding → audio embedding → fused-block stack with the video stream absent → audio output projection.

In [ ]:
with torch.no_grad():
    audio_reference = baseline(
        None, x_audio, audio_text, audio_pooled, audio_time,
        None, audio_rope, audio_text_rope,
    )
    audio_expanded_closed = expanded(
        None, x_audio, audio_text, audio_pooled, audio_time,
        None, audio_rope, audio_text_rope,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )

torch.testing.assert_close(audio_expanded_closed, audio_reference, rtol=0, atol=0)
print("Audio-only output:", tuple(audio_expanded_closed.shape))
print("Zero-gate parity through full DiffusionTransformer3D: passed")

## 7. Joint video+audio route through the full model

We now provide a tiny two-frame latent video and both modality-specific text/time inputs. This exercises Kandinsky's fused video/audio stack with game tokens forwarded into each block.

In [ ]:
VIDEO_FRAMES = 2
VIDEO_HEIGHT = 2
VIDEO_WIDTH = 2

x_video = torch.randn(
    BATCH, VIDEO_FRAMES, VIDEO_HEIGHT, VIDEO_WIDTH,
    BASE_CONFIG["in_visual_dim"],
)
video_text = torch.randn(BATCH, TEXT_LEN, BASE_CONFIG["in_text_dim"])
video_pooled = torch.randn(BATCH, BASE_CONFIG["in_text_dim2"])
video_time = torch.tensor([0.25, 0.65])

video_grid = (VIDEO_FRAMES, 1, 1)  # patch size (1, 2, 2) over a 2x2 latent frame
video_rope = expanded.visual_rope(
    video_grid,
    [torch.arange(VIDEO_FRAMES), torch.arange(1), torch.arange(1)],
)
video_text_rope = expanded.video_text_rope(text_positions)
joint_text = [video_text, audio_text]
joint_pooled = [video_pooled, audio_pooled]
joint_time = [video_time, audio_time]
joint_text_rope = [video_text_rope, audio_text_rope]

with torch.no_grad():
    joint_reference = baseline(
        x_video, x_audio, joint_text, joint_pooled, joint_time,
        video_rope, audio_rope, joint_text_rope,
    )
    joint_expanded_closed = expanded(
        x_video, x_audio, joint_text, joint_pooled, joint_time,
        video_rope, audio_rope, joint_text_rope,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )

torch.testing.assert_close(joint_expanded_closed[0], joint_reference[0], rtol=0, atol=0)
torch.testing.assert_close(joint_expanded_closed[1], joint_reference[1], rtol=0, atol=0)
print("Joint video output:", tuple(joint_expanded_closed[0].shape))
print("Joint audio output:", tuple(joint_expanded_closed[1].shape))
print("Zero-gate parity through full AV forward: passed")

## 8. Open only the final Game→Audio gate

The new modulation gates start at zero, so they initially preserve the base model. For a plumbing demonstration only, we set a nonzero bias on the final block's game gate. This is **not training** and does not represent a learned control.

In [ ]:
last_block = expanded.visual_transformer_blocks[-1]
with torch.no_grad():
    last_block.game_audio_modulation.out_layer.bias.fill_(0.1)

with torch.no_grad():
    audio_with_game = expanded(
        None, x_audio, audio_text, audio_pooled, audio_time,
        None, audio_rope, audio_text_rope,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )
    joint_with_game = expanded(
        x_video, x_audio, joint_text, joint_pooled, joint_time,
        video_rope, audio_rope, joint_text_rope,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )

audio_only_delta = (audio_with_game - audio_reference).abs().mean().item()
joint_audio_delta = (joint_with_game[1] - joint_reference[1]).abs().mean().item()
joint_video_delta = (joint_with_game[0] - joint_reference[0]).abs().mean().item()

assert audio_only_delta > 0
assert joint_audio_delta > 0
torch.testing.assert_close(joint_with_game[0], joint_reference[0], rtol=0, atol=0)

print("Audio-only mean output change:", audio_only_delta)
print("Joint audio mean output change:", joint_audio_delta)
print("Joint video mean output change:", joint_video_delta)
print("Game signal reaches audio; final-block video output is unchanged: passed")

## 9. Verify that padding is ignored

We change only the values of masked event tokens. Audio output should remain unchanged.

In [ ]:
changed_tokens = game_tokens.clone()
changed_tokens[~game_valid] += 1000.0

with torch.no_grad():
    masked_a = expanded(
        None, x_audio, audio_text, audio_pooled, audio_time,
        None, audio_rope, audio_text_rope,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )
    masked_b = expanded(
        None, x_audio, audio_text, audio_pooled, audio_time,
        None, audio_rope, audio_text_rope,
        game_tokens=changed_tokens,
        game_attn_mask=game_attn_mask,
    )

torch.testing.assert_close(masked_a, masked_b, rtol=1e-5, atol=1e-6)
print("Padded-token isolation through full model: passed")

## What Step 4 establishes — and what remains

**Established if every cell passes:** game tokens and a separate key mask reach Kandinsky's real full DiT forward method, through both audio-only and joint video+audio routes. The expanded architecture can load the old model's common weights with only new branch parameters absent, and a nonzero demo gate affects audio.

**Not established:** loading a real pretrained checkpoint, integration with Kandinsky's generation pipeline/configuration, training, audible quality, or useful learned control. The demonstration uses a tiny random model and manually opens the gate.

**Next:** decide the checkpoint-loading strategy and inspect how Kandinsky's pipeline constructs and invokes this DiT. Only then prepare an isolated inference prototype or a small training adapter.